# Jal-Saathi RL Training Pipeline on Kaggle (CPU)

This notebook runs offline PPO training for **Wheat** and **Paddy** irrigation environments.
- Installs `requirements-rl.txt`
- Unzips the uploaded repo dataset
- Links weather parquets from the uploaded weather dataset
- Runs `--smoke` test (validating reward-curve summary, evaluation table, ONNX parity with max abs diff $\le 10^{-4}$, and replay JSON in `docs/CONTRACTS.md` schema)
- Runs full PPO training
- Copies exported artifacts (`.onnx`, replay JSONs, `golden_obs.json`) to `/kaggle/working`

In [ ]:
# =============================================================================
# 1. Configuration & Timesteps
# =============================================================================
SMOKE_TIMESTEPS = 20000
FULL_TIMESTEPS = 100000
RUN_SMOKE = True
RUN_FULL = True
CROPS = ["wheat", "paddy"]

print(f"Smoke timesteps: {SMOKE_TIMESTEPS} | Full timesteps: {FULL_TIMESTEPS}")

In [ ]:
# =============================================================================
# 2. Unpack Repository Dataset
# =============================================================================
import os
import glob
import shutil
import zipfile

WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

# Look for repository zip in /kaggle/input
zip_candidates = glob.glob("/kaggle/input/**/*.zip", recursive=True)
repo_found = False

for zf in zip_candidates:
    try:
        with zipfile.ZipFile(zf, 'r') as zip_ref:
            namelist = zip_ref.namelist()
            if any("rl/train.py" in name for name in namelist) or any("train.py" in name for name in namelist):
                zip_ref.extractall(WORK_DIR)
                print(f"Extracted repo zip: {zf} into {WORK_DIR}")
                repo_found = True
                break
    except Exception as e:
        print(f"Skipping {zf}: {e}")

# Determine repo root
if os.path.exists(os.path.join(WORK_DIR, "Jal-Saathi")):
    REPO_ROOT = os.path.join(WORK_DIR, "Jal-Saathi")
elif os.path.exists(os.path.join(WORK_DIR, "rl", "train.py")):
    REPO_ROOT = WORK_DIR
else:
    # Check if repo was uploaded as an unzipped Kaggle dataset folder
    input_repos = glob.glob("/kaggle/input/**/rl/train.py", recursive=True)
    if input_repos:
        cand_root = os.path.abspath(os.path.join(os.path.dirname(input_repos[0]), ".."))
        REPO_ROOT = os.path.join(WORK_DIR, "Jal-Saathi")
        if not os.path.exists(REPO_ROOT):
            shutil.copytree(cand_root, REPO_ROOT)
        print(f"Copied repo from {cand_root} to {REPO_ROOT}")
    else:
        REPO_ROOT = WORK_DIR

print(f"Repository root: {REPO_ROOT}")
assert os.path.exists(os.path.join(REPO_ROOT, "rl", "train.py")), "rl/train.py not found!"

In [ ]:
# =============================================================================
# 3. Install Requirements
# =============================================================================
req_path = os.path.join(REPO_ROOT, "rl", "requirements-rl.txt")
!pip install -q -r {req_path}
print("Installed requirements-rl.txt successfully.")

In [ ]:
# =============================================================================
# 4. Link / Copy Weather Parquet Files
# =============================================================================
target_weather_dir = os.path.join(REPO_ROOT, "data", "processed")
os.makedirs(target_weather_dir, exist_ok=True)

# Find parquet files from Kaggle input datasets
weather_files = glob.glob("/kaggle/input/**/weather_*.parquet", recursive=True)

for wf in weather_files:
    dest = os.path.join(target_weather_dir, os.path.basename(wf))
    if not os.path.exists(dest) or os.path.abspath(wf) != os.path.abspath(dest):
        shutil.copy2(wf, dest)
        print(f"Copied: {os.path.basename(wf)} -> {dest}")

print("Weather parquet files in data/processed:")
print(os.listdir(target_weather_dir))

In [ ]:
# =============================================================================
# 5. Run Smoke Test (--smoke)
# =============================================================================
if RUN_SMOKE:
    for crop in CROPS:
        print(f"\n=========================================")
        print(f"  RUNNING --smoke PIPELINE: {crop.upper()}")
        print(f"=========================================\n")
        !cd {REPO_ROOT} && python rl/train.py --crop {crop} --smoke

In [ ]:
# =============================================================================
# 6. Run Full Training Run
# =============================================================================
if RUN_FULL:
    for crop in CROPS:
        print(f"\n=========================================")
        print(f"  RUNNING FULL TRAINING: {crop.upper()} ({FULL_TIMESTEPS} timesteps)")
        print(f"=========================================\n")
        !cd {REPO_ROOT} && python rl/train.py --crop {crop} --timesteps {FULL_TIMESTEPS}

In [ ]:
# =============================================================================
# 7. Copy Artifacts to /kaggle/working
# =============================================================================
artifact_folders = [
    os.path.join(REPO_ROOT, "artifacts"),
    os.path.join(REPO_ROOT, "rl", "artifacts")
]

copied_files = []
for af in artifact_folders:
    if os.path.exists(af):
        for fname in os.listdir(af):
            fpath = os.path.join(af, fname)
            if os.path.isfile(fpath):
                dest = os.path.join("/kaggle/working", fname)
                shutil.copy2(fpath, dest)
                copied_files.append(fname)
                print(f"Saved artifact: {fname}")

print("\nAll artifacts ready in /kaggle/working:")
for f in sorted(list(set(copied_files))):
    size = os.path.getsize(os.path.join("/kaggle/working", f))
    print(f"  - {f} ({size:,} bytes)")